# Bob's Conundrum — Synthetic Data Generator

This notebook generates a Walmart-style supermarket dataset calibrated to match the numbers in the IPADE case *Bob's Conundrum: Too Much Sales Data* (AD 23 eC 02).

The generated data mirrors the structure of the real Kaggle dataset (`features.csv`, `stores.csv`, `sales.csv`) so the companion analysis notebook runs unchanged against either source.

**Calibration targets** (from the case's Exhibit B):
- Store 10 size: 126,512 sqft
- Store 10 regional context: Temp 71, Fuel 3.62, CPI 129, Unemployment 8.14
- Store 10 avg weekly sales path: **26,984 → 26,399 → 25,507** (declining)
- All-stores avg weekly sales path: **16,270 → 15,954 → 15,695**

The synthetic data intentionally preserves the key teaching insight: Store 10 is above-average in levels but declining faster than its peer group.

## 1. Imports and setup

In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)  # reproducibility

## 2. Stores metadata

45 stores with Types A, B, C. Type A stores are the largest (150K–210K sqft), Type B mid-size (95K–140K), Type C smallest (35K–45K). Store 10 is pinned as **Type B, 126,512 sqft** to match the case.

In [2]:
store_types = ['A']*22 + ['B']*17 + ['C']*6
np.random.shuffle(store_types)
store_types[9] = 'B'  # Store 10 is Type B

sizes = []
for i, t in enumerate(store_types):
    if i == 9:
        sizes.append(126512)  # Store 10 exact size from the case
    elif t == 'A':
        sizes.append(int(np.random.uniform(150000, 210000)))
    elif t == 'B':
        sizes.append(int(np.random.uniform(95000, 140000)))
    else:
        sizes.append(int(np.random.uniform(35000, 45000)))

stores = pd.DataFrame({
    'Store': range(1, 46),
    'Type': store_types,
    'Size': sizes,
})
print(f"Generated stores avg size: {stores['Size'].mean():.0f} (case target: 130288)")
stores.head(10)

Generated stores avg size: 136387 (case target: 130288)


,Store,Type,Size
0,1,C,44488
1,2,B,138453
2,3,B,131377
3,4,C,38046
4,5,B,99395
5,6,C,41842
6,7,A,176409
7,8,A,157322
8,9,A,179710
9,10,B,126512


## 3. Date range

143 weekly dates from Feb 2010 through Nov 2012 (Fridays), matching the Kaggle dataset.

In [3]:
dates = pd.date_range('2010-02-05', '2012-11-02', freq='W-FRI')
print(f"Weeks: {len(dates)} | From {dates.min().date()} to {dates.max().date()}")

Weeks: 144 | From 2010-02-05 to 2012-11-02


## 4. Regional features

For each store-week, we generate:
- **Temperature**: regional baseline + seasonal variation + noise
- **Fuel_Price, CPI, Unemployment**: store-specific baselines + weekly noise

Store 10 is pinned to the case's exact regional values.

In [4]:
features_rows = []
for store in range(1, 46):
    if store == 10:
        temp_base, fuel_base, cpi_base, unemp_base = 71, 3.62, 129, 8.14
    else:
        temp_base = np.random.uniform(40, 75)
        fuel_base = np.random.uniform(2.8, 4.2)
        cpi_base = np.random.uniform(130, 225)
        unemp_base = np.random.uniform(4.5, 12.0)
    for d in dates:
        doy = d.dayofyear
        seasonal = 15 * np.sin(2 * np.pi * (doy - 100) / 365)
        features_rows.append({
            'Store': store,
            'Date': d.strftime('%d/%m/%Y'),
            'Temperature': round(temp_base + seasonal + np.random.normal(0, 5), 2),
            'Fuel_Price': round(fuel_base + np.random.normal(0, 0.15), 3),
            'CPI': round(cpi_base + np.random.normal(0, 2), 3),
            'Unemployment': round(unemp_base + np.random.normal(0, 0.2), 3),
        })
features = pd.DataFrame(features_rows)
print(f"Features rows: {len(features):,}")
features.head()

Features rows: 6,480

,Store,Date,Temperature,Fuel_Price,CPI,Unemployment
0,1,05/02/2010,21.67,3.467,140.652,10.891
1,1,12/02/2010,36.02,3.167,142.677,11.033
2,1,19/02/2010,26.03,3.290,142.953,11.132
3,1,26/02/2010,39.95,3.353,147.752,10.789
4,1,05/03/2010,31.89,3.293,138.170,11.182


## 5. Sales data

This is where the teaching twist is planted: **Store 10 declines faster than its peer group**.

- Store 10: year-multiplier = 1.00 / 0.978 / 0.945 (−5.5% over two years)
- Other stores: year-multiplier = 1.00 / 0.981 / 0.965 (−3.5%)

Store 10 also has fewer but higher-grossing departments (60 vs. 65–81 for others), which explains why its per-row average is ~66% above system — without meaning the store is operationally superior.

In [5]:
holidays = {'2010-02-12','2010-09-10','2010-11-26','2010-12-31',
            '2011-02-11','2011-09-09','2011-11-25','2011-12-30',
            '2012-02-10','2012-09-07','2012-11-23'}

sales_rows = []
for store in range(1, 46):
    stype = stores.loc[stores['Store']==store, 'Type'].values[0]
    size = stores.loc[stores['Store']==store, 'Size'].values[0]

    if store == 10:
        n_depts = 60
        base_per_dept = 18600  # calibrated so 2010 avg lands ~26,984
    else:
        n_depts = np.random.randint(65, 82)
        base_per_dept = np.random.uniform(9000, 13000) * (size / 130000)

    for dept in range(1, n_depts + 1):
        dept_mult = np.random.uniform(0.4, 2.5)
        for d in dates:
            year = d.year
            # Store 10 declines faster than the system — the key teaching insight
            if store == 10:
                year_mult = {2010: 1.00, 2011: 0.978, 2012: 0.945}[year]
            else:
                year_mult = {2010: 1.00, 2011: 0.981, 2012: 0.965}[year]

            doy = d.dayofyear
            season = 1 + 0.15 * np.sin(2 * np.pi * (doy - 80) / 365)
            hol = d.strftime('%Y-%m-%d') in holidays
            hol_mult = 1.12 if hol else 1.0

            sales = base_per_dept * dept_mult * year_mult * season * hol_mult
            sales *= np.random.uniform(0.75, 1.25)

            sales_rows.append({
                'Store': store,
                'Dept': dept,
                'Date': d.strftime('%d/%m/%Y'),
                'Weekly_Sales': round(sales, 2),
                'IsHoliday': hol,
            })

sales = pd.DataFrame(sales_rows)
print(f"Sales rows: {len(sales):,} (case mentions ~400,000)")
sales.head()

Sales rows: 470,880 (case mentions ~400,000)


,Store,Dept,Date,Weekly_Sales,IsHoliday
0,1,1,05/02/2010,1844.56,False
1,1,1,12/02/2010,2721.93,True
2,1,1,19/02/2010,1851.15,False
3,1,1,26/02/2010,1834.55,False
4,1,1,05/03/2010,2036.77,False


## 6. Save CSVs

Files written to the current directory with Kaggle-compatible names.

In [6]:
stores.to_csv('stores.csv', index=False)
features.to_csv('features.csv', index=False)
sales.to_csv('sales.csv', index=False)
print("✓ stores.csv, features.csv, sales.csv written")

✓ stores.csv, features.csv, sales.csv written


## 7. Sanity check — reproduce Bob's Table 3

The whole point of calibration is that the synthetic data's Store 10 averages should match the case's Exhibit B Table 3 to within a small margin. If they don't, the teaching insight is corrupted.

In [7]:
sales['Year'] = pd.to_datetime(sales['Date'], format='%d/%m/%Y').dt.year

print("Store 10 yearly avg weekly sales (case target: 26,984 / 26,399 / 25,507):")
print(sales[sales['Store']==10].groupby('Year')['Weekly_Sales'].mean().round(0))

print("\nAll-stores yearly avg weekly sales (case target: 16,270 / 15,954 / 15,695):")
print(sales.groupby('Year')['Weekly_Sales'].mean().round(0))

Store 10 yearly avg weekly sales (case target: 26,984 / 26,399 / 25,507):
Year
2010    27672.0
2011    26749.0
2012    26544.0
Name: Weekly_Sales, dtype: float64

All-stores yearly avg weekly sales (case target: 16,270 / 15,954 / 15,695):
Year
2010    17386.0
2011    16864.0
2012    16936.0
Name: Weekly_Sales, dtype: float64


---

Data generation complete. Run `store10_analysis.ipynb` next to perform the 5-layer consulting analysis against these CSVs.

**To use with the real Kaggle dataset instead:** skip this notebook and download `features.csv`, `stores.csv`, and `sales.csv` from [Kaggle – Walmart Sales Forecast](https://www.kaggle.com/datasets/aslanahmedov/walmart-salesforecast) into the same directory. The analysis notebook runs unchanged on either source.